# Mini-proyecto: "Zaragoza chico"**Bloque 5 de la clase · 25 minutos**Vas a recorrer los **cinco pasos** del proyecto real sobre **una sola página lógica**de las bases de la Municipalidad Provincial de Ilo:| Paso | Qué hace | Min ||---|---|---|| 1. Triaje | confirma que la página es imagen | 2 || 2. Preparación | rasteriza, parte la hoja en dos, endereza | 6 || 3. OCR | texto, cajas, confianza | 5 || 4. Extracción | cuatro campos a JSON | 8 || 5. Entrega | CSV y tabla en pantalla | 4 |Cuando al final de la clase veas la aplicación real de 110 páginas, vas a reconocercada una de estas piezas.> **La idea única de hoy:** conocer tu hardware decide qué modelo puedes correr, y> arreglar la imagen rinde más que traer un modelo más grande.

## Paso 0 — Dónde estamos corriendoAntes de nada: ¿hay GPU o no? Esto no es curiosidad, **decide qué nivel de modelopuedes usar**.El `doctor` hace seis chequeos. El sexto es el único que no se puede falsificar:carga el modelo de verdad y mira si la VRAM se mueve.

In [ ]:
# Nos aseguramos de correr desde la raíz del proyecto, no desde notebooks/
import os, sys
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

print("Carpeta de trabajo:", Path.cwd())

In [ ]:
from src.comun import cargar_config
from src import doctor

config = cargar_config()
chequeos = doctor.diagnosticar(config)

for c in chequeos:
    print(c.linea())

hay_gpu = chequeos[3].ok and chequeos[5].ok
print()
print("MODO:", "GPU" if hay_gpu else "CPU")

### Si te salió CPU, no pasa nadaEl notebook sigue funcionando. Pero bajamos el nivel del modelo a `tiny` para que eltiempo sea tolerable: 1.5 millones de parámetros en vez de 34.5 millones.Eso **es** la lección de la clase, en vivo: el hardware decide qué modelo corres.

In [ ]:
if not hay_gpu:
    config["ocr"]["device"] = "cpu"
    config["ocr"]["nivel"] = "tiny"
    print("Sin GPU: se usa nivel 'tiny' en CPU para que el tiempo sea tolerable.")
    print("Precisión de reconocimiento: 73.5% con tiny, contra 83.2% con medium.")
else:
    print(f"Con GPU: se usa nivel '{config['ocr']['nivel']}' en GPU.")

---## Paso 1 — Triaje: ¿esta página necesita OCR?**La regla de oro:** usar OCR **solo** en las páginas que son imagen. Las que tienencapa de texto se extraen gratis e instantáneamente.> Analogía: el triaje es la enfermera que decide quién pasa a emergencia. No todos los> pacientes necesitan cirugía. El OCR es la cirugía: cara y lenta.**Pregunta para pensar mientras corre:** si de un documento de 500 páginas solo 20 sonescaneadas, ¿cuánto te ahorras haciendo triaje primero?

In [ ]:
from src import triaje

filas = triaje.ejecutar(config)

# Las primeras cinco hojas, para ver la forma del dato
for fila in filas[:5]:
    print(f"hoja {fila.pagina:>3} | {fila.caracteres:>4} caracteres | "
          f"{fila.imagenes_embebidas:>3} imágenes embebidas | {fila.clase}")

### Lee el resultado con cuidadoDos cosas:1. **0 caracteres en las 57 hojas.** Cero por ciento aprovechable sin OCR. Este   documento es 100% foto.2. **Entre 9 y 63 imágenes embebidas por hoja.** Esa es la pista de que el escáner   guardó cada hoja partida en mosaicos.> El PDF no guarda una foto de la hoja, guarda el **rompecabezas desarmado**. Si pides> "las imágenes", te dan las piezas sueltas. Hay que pedir la foto del rompecabezas> armado — y eso es lo que hace el renderizador en el paso 2.

---## Paso 2 — Preparación: **aquí se gana la pelea**Tres cosas, en este orden, y el orden no es negociable:1. **Rasterizar** la hoja completa a un DPI elegido.2. **Partir** la hoja en dos páginas lógicas (este PDF viene a dos por hoja).3. **Corregir la orientación** (este PDF viene girado).Partir *después* del OCR no sirve: el modelo ya mezcló las dos páginas.### Primero, el error: OCR sin prepararVamos a hacer OCR sobre la hoja **cruda**, sin partir ni enderezar, para ver qué sale.

In [ ]:
import pypdfium2 as pdfium
from src.comun import ruta_absoluta

ruta_pdf = ruta_absoluta(config["rutas"]["pdf_entrada"])
procesados = ruta_absoluta(config["rutas"]["procesados"])
procesados.mkdir(parents=True, exist_ok=True)

# Hoja 1 completa, sin partir y sin rotar: el "antes"
documento = pdfium.PdfDocument(str(ruta_pdf))
hoja_cruda = documento[0].render(scale=425 / 72).to_pil()
documento.close()

ruta_cruda = procesados / "demo_sin_preparar.png"
hoja_cruda.save(ruta_cruda)

print(f"Hoja cruda: {hoja_cruda.size[0]}x{hoja_cruda.size[1]} px")
hoja_cruda.resize((hoja_cruda.size[0] // 6, hoja_cruda.size[1] // 6))

Míralo bien: son **dos páginas** en una hoja, y están **de costado**.Ahora el OCR sobre esa imagen tal cual.

In [ ]:
from src import ocr_run

# Un objeto PaddleOCR, construido UNA sola vez y reutilizado todo el notebook.
# Reconstruirlo por página recarga el modelo y arruina cualquier medición.
motor = ocr_run.construir_ocr(config)

lineas_mal = ocr_run.ocr_imagen(ruta_cruda, config, motor)

print(f"{len(lineas_mal)} líneas reconocidas\n")
for linea in lineas_mal[:12]:
    print(f"  {linea.confianza:.3f}  {linea.texto}")

### Ahora con la imagen preparadaMismo modelo. Misma máquina. Mismo todo. Solo cambia que la imagen está derecha ypartida en una página por archivo.

In [ ]:
from src import preparar

# Hojas 1 Y 2, no solo la 1. Verificado sobre el documento: la entidad esta en el
# encabezado de la hoja 1, pero el objeto, el numero y el departamento estan en la
# caratula interna, que es la hoja 2. Con una sola hoja, dos de los cuatro campos
# salen vacios por mas que el modelo sea bueno: no estan en el texto.
generadas = preparar.ejecutar(hoja_inicial=1, hoja_final=2, config=config, forzar=True)

for ruta in generadas:
    print(ruta.name)

In [ ]:
from PIL import Image

# La primera página lógica de la hoja 1, ya derecha
pagina = Image.open(generadas[0])
print(f"{generadas[0].name}: {pagina.size[0]}x{pagina.size[1]} px")
pagina.resize((pagina.size[0] // 4, pagina.size[1] // 4))

In [ ]:
lineas_bien = ocr_run.ocr_imagen(generadas[0], config, motor)

print(f"{len(lineas_bien)} líneas reconocidas\n")
for linea in lineas_bien[:15]:
    print(f"  {linea.confianza:.3f}  {linea.texto}")

### Compara los dos resultados```Sin preparar:  letras sueltas, confianza en el pisoPreparado:     texto legible, confianza alta```> **No cambiamos de modelo. Es el mismo modelo, con la imagen arreglada.**Hay un dato publicado que lo confirma: cuando a estos modelos se les dan **fotos** dedocumentos en vez de escaneos limpios, el mejor de todos cae de **91.9 a 72.2** puntos.Veinte puntos. No hay modelo que compense una mala imagen.

In [ ]:
import statistics

def resumen(lineas, etiqueta):
    if not lineas:
        print(f"{etiqueta:<16} sin líneas")
        return
    confianzas = [l.confianza for l in lineas]
    bajas = sum(1 for c in confianzas if c < config["ocr"]["umbral_confianza"])
    print(f"{etiqueta:<16} {len(lineas):>4} líneas | "
          f"confianza media {statistics.mean(confianzas):.3f} | "
          f"{bajas} bajo el umbral ({bajas/len(lineas)*100:.0f}%)")

resumen(lineas_mal, "sin preparar")
resumen(lineas_bien, "preparado")

---## Paso 3 — Qué devuelve el OCR, de verdadEl OCR **no** devuelve texto. Devuelve **texto + geometría + confianza**, y los tresdatos valen igual.| Campo | Qué es | Para qué sirve ||---|---|---|| `texto` | la cadena reconocida | lo obvio || `caja` | cuatro puntos en la imagen | resaltar en el original lo que la IA afirma || `confianza` | número entre 0 y 1 | decidir qué revisa un humano |> Analogía: no es un lector que te cuenta el libro. Es un lector que además te señala> con el dedo en qué parte de la página está cada frase, y te dice qué tan seguro está> de cada una.

In [ ]:
primera = lineas_bien[0]

print("texto     :", primera.texto)
print("confianza :", primera.confianza)
print("caja      :", primera.caja)

### Esto desbloquea la idea más valiosa de la claseLa **confianza** permite escalar por costo: corres el modelo barato en todo, y soloreenvías al modelo caro las páginas donde el barato dudó.> Analogía: el profesor corrige rápido todos los exámenes y solo vuelve a revisar con> calma los que quedaron entre 10 y 12. No pierde tiempo con los de 18 ni con los de 5.

In [ ]:
umbral = config["ocr"]["umbral_confianza"]
dudosas = [l for l in lineas_bien if l.confianza < umbral]

print(f"{len(dudosas)} de {len(lineas_bien)} líneas por debajo de {umbral}")
print("Estas son las que convendría que lea un humano:\n")
for linea in dudosas[:8]:
    print(f"  {linea.confianza:.3f}  {linea.texto}")

---## Paso 4 — Extracción: de texto a datosCuatro campos. Dos formas de sacarlos, y **la comparación entre las dos es la lección**:- **Vía A, reglas.** Expresiones regulares. Gratis, instantánea, frágil ante una  errata del OCR.- **Vía B, modelo de lenguaje.** Se le pasa el texto y se le pide JSON. Tolera erratas,  cuesta tokens.### Un detalle que descubrimos leyendo el documento realEl nombre de la entidad **no está en la carátula**. Vive en el encabezado de la hojafísica, que al partirla cae en la *otra* página lógica. Por eso la extracción procesa**las dos mitades** de la hoja, no solo la carátula.Y el encabezado está cortado en el escaneo: donde dice `PROVINCIAL`, el OCR lee`ROVINCIAL`. Las expresiones regulares tienen que tolerarlo o fallan.

### La clave de API no se escribe en el notebookNunca. Ni "solo para probar". Una clave escrita en una celda se guarda dentro del`.ipynb`, el `.ipynb` se sube al repositorio, y una clave que llegó a un repositoriopúblico **queda en el historial de git para siempre**. Borrarla en un commit posteriorno la saca del historial: hay que revocarla y generar otra.El proyecto la lee de un archivo `.env` que `.gitignore` excluye:```copy .env.example .env      # y pega tu clave dentro del .env````src/comun.py` lo carga solo al importar cualquier módulo. No hay que hacer nada más.

In [ ]:
import os

# comun.py ya cargo el .env al importarse. Esto solo comprueba que la clave llego.
# NO se imprime ningun fragmento de la clave: lo que se imprime queda guardado
# dentro del .ipynb y viaja al repositorio con el.
HAY_CLAVE = bool(os.environ.get("ANTHROPIC_API_KEY"))

if HAY_CLAVE:
    print("Clave cargada desde .env. Via B disponible.")
else:
    print("No hay ANTHROPIC_API_KEY.")
    print()
    print("Para la via B:  copy .env.example .env   y pega tu clave dentro.")
    print("El notebook sigue funcionando sin ella: la via A no necesita API.")

In [ ]:
# OCR de las cuatro mitades (dos hojas x dos paginas logicas)
resultados = ocr_run.ocr_lote(generadas, config, forzar=True)

for ruta, lineas in resultados.items():
    print(f"{ruta.name}: {len(lineas)} lineas")

In [ ]:
from src import extraer

# Sin el argumento `hojas`: usa extraccion.hojas_metadatos de config.yaml, que es
# [1, 2]. Ahi estan los cuatro campos.
texto_ocr = extraer.texto_de_hojas(config)
print(f"{len(texto_ocr)} caracteres de texto OCR\n")
print(texto_ocr[:700])

In [ ]:
campos = config["extraccion"]["campos"]

por_reglas = extraer.extraer_por_reglas(texto_ocr, campos)

for nombre, campo in por_reglas.campos.items():
    marca = "SIN VERIFICAR" if campo.sin_verificar else "verificado"
    print(f"{nombre:<22} [{marca}]")
    print(f"  {campo.valor}\n")

### La regla de verificaciónEsta regla viene del proyecto real y es la parte que de verdad importa:> Todo dato extraído debe existir **literalmente** en el texto del OCR. Si no existe,> se marca como `sin_verificar`.Un modelo que "corrige" el número de licitación y acierta es **tan peligroso** como unoque lo inventa, porque la próxima vez va a corregir y fallar.Ejemplo real de este documento: contiene la errata **"ASENTAMIENTO HUAMANO"**. Es de laMunicipalidad, no del OCR. Un modelo servicial la "arreglaría" a HUMANO — y con eso eldato dejaría de coincidir con el documento oficial.

In [ ]:
# La verificación en acción
print("¿'Municipalidad Provincial de Ilo' está literal en el texto OCR?")
print("  ", extraer.verificar_literal("Municipalidad Provincial de Ilo", texto_ocr))

print("\n¿Y una invención plausible pero falsa?")
print("  ", extraer.verificar_literal("Municipalidad Provincial de Tacna", texto_ocr))

In [ ]:
# Via B: el modelo de lenguaje. Usa la clave del .env, cargada mas arriba.
por_modelo = extraer.extraer_por_modelo(texto_ocr, campos) if HAY_CLAVE else None

if por_modelo is None:
    print("Via B no disponible: falta la clave. La via A ya corrio.")
else:
    for nombre, campo in por_modelo.campos.items():
        marca = "SIN VERIFICAR" if campo.sin_verificar else "verificado"
        print(f"{nombre:<22} [{marca}]")
        print(f"  {campo.valor}\n")

---## Paso 5 — Entrega: CSV y tablaEl entregable de la sesión. En el proyecto real esto es un informe LaTeX membretado; enescala de clase es un CSV y una tabla en pantalla. **Los pasos son los mismos.**

In [ ]:
extracciones = [por_reglas] + ([por_modelo] if por_modelo else [])

destino = ruta_absoluta(config["rutas"]["salidas"])
ruta_json, ruta_csv = extraer.escribir_salidas(extracciones, destino)

extraer.imprimir_comparacion(extracciones, campos)
print(f"\nEntregable: {ruta_csv}")

In [ ]:
import pandas as pd

tabla = pd.read_csv(ruta_csv)
tabla

---## Tu entregable1. **El CSV** con los cuatro campos extraídos.2. **Una línea de respuesta:** ¿en qué modo corriste, y cuántos segundos por página te   tomó?Esa segunda parte es la que importa: te obliga a haber **medido**.

In [ ]:
import time

t0 = time.perf_counter()
_ = ocr_run.ocr_imagen(generadas[0], config, motor)
segundos = time.perf_counter() - t0

print(f"Modo               : {config['ocr']['device']}")
print(f"Nivel del modelo   : {config['ocr']['nivel']}")
print(f"Segundos por página: {segundos:.2f}")
print()
print("Ojo: esta medición es de UNA página, con el modelo ya cargado.")
print("Para medir en serio hace falta el protocolo de bench.py:")
print("  calentamiento descartado, varias páginas, y MEDIANA en vez de promedio.")

---## Lo que acabas de construir| Paso | En este notebook | En el proyecto real ||---|---|---|| Triaje | 57 hojas clasificadas | 110+ páginas de bases || Preparación | 1 hoja → 2 páginas derechas | el documento completo || OCR | 1 página lógica | capítulos III y IV completos || Extracción | 4 campos a JSON | un agente legal que redacta observaciones || Entrega | un CSV | un informe LaTeX membretado |**Los cinco pasos son los mismos.** Solo cambia la escala.### Las seis lecciones transferibles1. Definir el problema antes de elegir la herramienta.2. Conocer los límites de hardware: deciden qué es posible.3. La herramienta del tamaño del problema. No traer un cañón para matar una mosca.4. Híbrido: lo barato y masivo en local, el modelo caro solo donde agrega valor.5. Iterar y diagnosticar. Los fracasos enseñan.6. Humano al centro. La IA propone, la persona decide.### Sigue con- `fallas/README.md` — las tres fallas sembradas. Diagnostica leyendo el log.- `python -m src.bench rapido` — mide CPU contra GPU con protocolo honesto.